# 05 — Final results and presentation tables

**Workflow version:** 0.5.0

Generate clean tables and figures from completed local analyses. This notebook does not rerun the expensive phenotype benchmark.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
RESULTS_DIR = ROOT / "results"

required = [
    RESULTS_DIR / "02_benchmark_summary.csv",
    RESULTS_DIR / "02_pairwise_comparison.csv",
    RESULTS_DIR / "04_paper_vs_reproduced_summary.csv",
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing analysis outputs: " + ", ".join(path.name for path in missing))

summary = pd.read_csv(RESULTS_DIR / "02_benchmark_summary.csv")
pairwise = pd.read_csv(RESULTS_DIR / "02_pairwise_comparison.csv")
paper_comparison = pd.read_csv(RESULTS_DIR / "04_paper_vs_reproduced_summary.csv")

In [ ]:
presentation_summary = paper_comparison[[
    "model", "correct", "n", "accuracy",
    "paper_correct", "paper_n", "paper_accuracy",
    "correct_delta", "accuracy_delta",
]].copy()

presentation_summary["accuracy_percent"] = 100 * presentation_summary["accuracy"]
presentation_summary["paper_accuracy_percent"] = 100 * presentation_summary["paper_accuracy"]

display(presentation_summary)
presentation_summary.to_csv(RESULTS_DIR / "05_presentation_summary.csv", index=False)

In [ ]:
class_counts = summary.set_index("model")[["correct", "type_I", "type_II"]]
ax = class_counts.plot(kind="bar")
ax.set_title("Auxotrophy benchmark classification")
ax.set_xlabel("Model")
ax.set_ylabel("Gene-compound pairs")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "05_classification_counts.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
accuracy_table = presentation_summary.set_index("model")[["accuracy_percent", "paper_accuracy_percent"]]
ax = accuracy_table.plot(kind="bar")
ax.set_title("Reproduced versus reported accuracy")
ax.set_xlabel("Model")
ax.set_ylabel("Accuracy (%)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "05_accuracy_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
change_counts = pairwise["change"].value_counts().rename_axis("change").to_frame("n")
display(change_counts)
change_counts.to_csv(RESULTS_DIR / "05_change_counts.csv")

fixed_or_regressed = pairwise[pairwise["change"].isin(["fixed", "regression"])].copy()
fixed_or_regressed.to_csv(RESULTS_DIR / "05_fixed_or_regressed_pairs.csv", index=False)
print("Final presentation outputs saved to:", RESULTS_DIR)

## Presentation note

Use this notebook for clean figures and tables only after the strict benchmark has completed without unresolved solver/input errors. Keep provisional exploratory results out of the final presentation.